# Training Pipeline (with constitution)

Builds a **training** dataset for a constitutional classifier. The one structural difference
from the eval pipeline is the **constitution**: Claude Opus generates a category hierarchy
(4 severity types), which seeds input generation. Then outputs → jailbreaks → merged dataset.

Single `data_dir` root; models default to their **role** (`None`); `resume=True` continues,
`resume=False` restarts a stage clean.

In [ ]:
from redact import (
    generate_constitution, generate_inputs, generate_outputs,
    generate_jailbreaks, build_dataset,
)

DATA_DIR = "./runs/training"   # one working root for the whole run
GEN_MODEL = None               # None -> uncensored_gen role default
CONST_MODEL = None             # None -> constitution_gen role default (Claude Opus)
NUM_TAXONOMY_CATEGORIES = 3
SAMPLES_PER_ENTRY = 3

## Step 0 — Constitution (Claude Opus)

In [ ]:
constitution = generate_constitution(
    data_dir=DATA_DIR,
    model=CONST_MODEL,
    num_categories=10,
    num_taxonomy_categories=NUM_TAXONOMY_CATEGORIES,
)
print(f"{len(constitution)} constitution entries")
constitution["entry_type"].value_counts()

## Step 1 — Inputs (constitution-seeded)

Passing `constitution_df=` triggers constitution-seeded mode. The inputs auto-route to
`{data_dir}/Datasets/constitution_inputs/` — no need to hand-manage the path.

In [ ]:
inputs = generate_inputs(
    data_dir=DATA_DIR,
    constitution_df=constitution,
    samples_per_entry=SAMPLES_PER_ENTRY,
    model=GEN_MODEL,
    style="long",
)
print(f"{len(inputs)} accepted prompts")
inputs.head()

## Step 2 — Outputs

In [ ]:
outputs = generate_outputs(
    inputs=inputs, data_dir=DATA_DIR, model=GEN_MODEL, max_per_category=20,
)
print(f"{len(outputs)} responses")
outputs.head()

## Step 3 — Jailbreak augmentation

In [ ]:
jailbreaks = generate_jailbreaks(
    inputs=inputs,
    data_dir=DATA_DIR,
    model=GEN_MODEL,
    include_translation=False,
    entry_types=["harmful", "dual_use_harmful"],
)
print(f"{len(jailbreaks)} jailbreak rows")
jailbreaks["technique"].value_counts().head(10)

## Step 4 — Merge into the training dataset

In [ ]:
dataset = build_dataset(data_dir=DATA_DIR)
dataset["dataset_type"].value_counts()

## Alternative — one-shot, config-driven

The `constitution` stage runs first and its entries seed `inputs` automatically. Each stage
writes a `{stage}.run.json` manifest under `{data_dir}/Datasets/`.

In [ ]:
from redact import run_pipeline

summary = run_pipeline(
    {
        "dataset_type": "training",
        "data_dir": DATA_DIR,
        "stages": ["constitution", "inputs", "outputs", "jailbreaks", "build"],
        "models": {"gen": GEN_MODEL, "constitution": CONST_MODEL},
        "augmentations": {"include_translation": False},
    },
    params={
        "constitution": {"num_taxonomy_categories": NUM_TAXONOMY_CATEGORIES},
        "inputs": {"samples_per_entry": SAMPLES_PER_ENTRY},
        "outputs": {"max_per_category": 20},
        "jailbreaks": {"entry_types": ["harmful", "dual_use_harmful"]},
    },
)
summary